<a href="https://colab.research.google.com/github/NULEVbIY/DLS-Face-Recognition-progect/blob/main/Zadanie_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ноутбук для задания 2

In [42]:
import zipfile
import os
import pandas as pd
from collections import defaultdict, Counter
import google.colab.drive as drive

drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/Face_Recognition_DLS/Final_Aligned_Dataset.zip"
IDENTITY_PATH = "/content/drive/MyDrive/Face_Recognition_DLS/identity_CelebA.txt"

identity_df = pd.read_csv(IDENTITY_PATH, sep=r"\s+", header=None, names=['image_id', 'identity_id'])

image_to_person = dict(zip(identity_df['image_id'], identity_df['identity_id']))

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    all_files = zip_ref.namelist()

    image_files = [os.path.basename(f) for f in all_files
                   if f.lower().endswith('.jpg') and os.path.basename(f)]
    image_files = list(set(image_files))
    image_files.sort()


person_images = defaultdict(int)

for img_file in image_files:
    if img_file in image_to_person:
        person_id = image_to_person[img_file]
        person_images[person_id] += 1

person_counts = dict(person_images)
total_images = sum(person_counts.values())

stats_df = pd.DataFrame([
    {'person_id': pid, 'count': cnt}
    for pid, cnt in sorted(person_counts.items(), key=lambda x: x[1], reverse=True)
])
stats_df.to_csv('/content/dataset_stats.csv', index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1

In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import io
import zipfile
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict
import google.colab.drive as drive

ZIP_PATH = "/content/drive/MyDrive/Face_Recognition_DLS/Final_Aligned_Dataset.zip"
IDENTITY_PATH = "/content/drive/MyDrive/Face_Recognition_DLS/identity_CelebA.txt"
STATS_PATH = "/content/dataset_stats.csv"

BATCH_SIZE = 64
EPOCHS = 50
LR = 0.001
EMBEDDING_DIM = 512
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


2

In [44]:
import os
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import pandas as pd

!rm -rf /content/dataset
!unzip -q "/content/drive/MyDrive/Face_Recognition_DLS/Final_Aligned_Dataset.zip" -d "/content/dataset"
!ls /content/dataset

class SimpleCelebADataset(Dataset):
    def __init__(self, folder_path, identity_path, transform=None, min_images=5):
        self.folder_path = folder_path
        self.transform = transform

        identity_df = pd.read_csv(identity_path, sep=r"\s+", header=None, names=['image_id', 'identity_id'])
        image_to_person = dict(zip(identity_df['image_id'], identity_df['identity_id']))

        stats_df = pd.read_csv(STATS_PATH)
        good_people = set(stats_df[stats_df['count'] >= min_images]['person_id'])

        self.samples = []
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith('.jpg'):
                    img_name = file
                    if img_name in image_to_person and image_to_person[img_name] in good_people:
                        person_id = image_to_person[img_name]
                        self.samples.append((os.path.join(root, img_name), person_id))

        unique_persons = sorted(list(set(p[1] for p in self.samples)))
        self.person_to_class = {pid: i for i, pid in enumerate(unique_persons)}
        self.samples = [(path, self.person_to_class[pid]) for path, pid in self.samples]
        self.num_classes = len(unique_persons)


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, class_id = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, class_id

train_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = SimpleCelebADataset(
    folder_path="/content/dataset",
    identity_path=IDENTITY_PATH,
    transform=train_transform
)

indices = list(range(len(dataset)))
np.random.seed(42)
np.random.shuffle(indices)
split_idx = int(0.8 * len(dataset))
train_idx = indices[:split_idx]
val_idx = indices[split_idx:]

train_dataset = torch.utils.data.Subset(dataset, train_idx)
val_dataset = torch.utils.data.Subset(dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

NUM_CLASSES = dataset.num_classes


Final_Aligned_Dataset


3

In [45]:
import torchvision.models as models

class SuperFaceNet(nn.Module):
    def __init__(self, num_classes=1000, embedding_dim=512):
        super().__init__()

        backbone = models.resnet50(pretrained=True)
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, embedding_dim)
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, return_embeddings=False):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        embeddings = self.embedding_head(x)
        return embeddings if return_embeddings else self.classifier(embeddings)

model_ce_final = SuperFaceNet(NUM_CLASSES).to(DEVICE)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [46]:
class ArcFaceLayer(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=1000, s=30.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings)
        weights = F.normalize(self.weight)

        cos_theta = torch.mm(embeddings, weights.t())

        target_theta = torch.acos(cos_theta[range(len(labels)), labels].clamp(-1+1e-7, 1-1e-7))
        target_cos = torch.cos(target_theta + self.m).clamp(-1+1e-7, 1-1e-7)
        cos_theta[range(len(labels)), labels] = target_cos

        return self.s * cos_theta

arcface_layer = ArcFaceLayer(EMBEDDING_DIM, NUM_CLASSES).to(DEVICE)


In [47]:
def train_epoch_ce(model, loader, criterion, optimizer):
    model.train()
    total_loss = correct = total = 0

    pbar = tqdm(loader, desc="CE Train")
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        pred = logits.argmax(1)
        correct += (pred == labels).sum().item()
        total += len(labels)

        pbar.set_postfix({
            'loss': f"{loss.item():.3f}",
            'acc': f"{correct/total:.1%}"
        })

    return total_loss/len(loader), correct/total

def train_epoch_arcface(model, arcface_layer, loader, criterion, optimizer_model, optimizer_arcface):
    model.train()
    arcface_layer.train()
    total_loss = correct = total = 0

    pbar = tqdm(loader, desc="ArcFace Train")
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer_model.zero_grad()
        optimizer_arcface.zero_grad()

        embeddings = model(images, return_embeddings=True)
        logits = arcface_layer(embeddings, labels)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_model.step()
        optimizer_arcface.step()

        total_loss += loss.item()
        pred = logits.argmax(1)
        correct += (pred == labels).sum().item()
        total += len(labels)

        pbar.set_postfix({
            'loss': f"{loss.item():.3f}",
            'acc': f"{correct/total:.1%}"
        })

    return total_loss/len(loader), correct/total

def validate(model, loader, use_arcface=False, arcface_layer=None):
    model.eval()
    total_loss = correct = total = 0

    with torch.no_grad():
        pbar = tqdm(loader, desc="Validation")
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            if use_arcface and arcface_layer:
                embeddings = model(images, return_embeddings=True)
                logits = arcface_layer(embeddings, labels)
            else:
                logits = model(images)

            loss = F.cross_entropy(logits, labels)
            total_loss += loss.item()
            pred = logits.argmax(1)
            correct += (pred == labels).sum().item()
            total += len(labels)

            pbar.set_postfix({'acc': f"{correct/total:.1%}"})

    return total_loss/len(loader), correct/total


In [28]:
criterion = nn.CrossEntropyLoss()

optimizer_ce = torch.optim.AdamW(model_ce_final.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_ce,
    max_lr=5e-3,
    epochs=50,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos'
)

history_ce_final = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc = 0

import os
if os.path.exists('/content/best_ce_70percent.pth'):
    os.remove('/content/best_ce_70percent.pth')
if os.path.exists('/content/final_ce_70percent.pth'):
    os.remove('/content/final_ce_70percent.pth')

for epoch in range(50):
    train_loss, train_acc = train_epoch_ce(model_ce_final, train_loader, criterion, optimizer_ce)
    val_loss, val_acc = validate(model_ce_final, val_loader)

    history_ce_final['train_loss'].append(train_loss)
    history_ce_final['train_acc'].append(train_acc)
    history_ce_final['val_loss'].append(val_loss)
    history_ce_final['val_acc'].append(val_acc)

    scheduler.step()

    print(f"Epoch {epoch+1}/50 | Train: {train_acc:.1%} | Val: {val_acc:.1%} | LR: {scheduler.get_last_lr()[0]:.1e}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_ce_final.state_dict(), '/content/best_ce_70percent_NEW.pth')
        print(f"✅ BEST: {best_acc:.1%}")

    if best_acc > 0.70:
        break

print(f"\nРЕЗУЛЬТАТ: {best_acc:.1%}")
torch.save(model_ce_final.state_dict(), '/content/final_ce_70percent_NEW.pth')


Validation: 100%|██████████| 47/47 [00:07<00:00,  6.36it/s, acc=16.2%]


Epoch 1/50 | Train: 16.5% | Val: 16.2% | LR: 2.0e-04
✅ BEST: 16.2%


Validation: 100%|██████████| 47/47 [00:06<00:00,  6.75it/s, acc=19.7%]


Epoch 2/50 | Train: 26.7% | Val: 19.7% | LR: 2.0e-04
✅ BEST: 19.7%


Validation: 100%|██████████| 47/47 [00:06<00:00,  6.75it/s, acc=23.5%]


...

Epoch 17/50 | Train: 86.7% | Val: 43.8% | LR: 2.0e-04


Validation: 100%|██████████| 47/47 [00:06<00:00,  7.05it/s, acc=43.9%]


Epoch 18/50 | Train: 87.0% | Val: 43.9% | LR: 2.0e-04


CE Train:  30%|██▉       | 56/187 [00:15<00:36,  3.60it/s, loss=0.292, acc=89.3%]


KeyboardInterrupt: 

Да уж обучение прошло так себе, модель получилась слабая, до 0.7 дойти не получилось, что только не крутил, не менял - не смог достичь нужного результата. Я думаю это из-за моего датасета, уж слабо я его отобрал (нужно было учесть разнообразие классов и подумать над другими атрибутами) и сделать ресайз на больший размер, хотя бы до 256 на 256. Также плохой оказалась реализация выравнивания, но ее я подробно описал в задании 1, напомню только, что ее работа ограничивалась лишь простыми случаями, что-то сложное сразу приводило к необучаемой картинке (например, глаза обрубало и был только рот). И сама модель переобучилась и не показала нужный результат, могу выделить, что была слабая аугментация изображений, что могло сплюсоваться с плохим выравниваем и привести к такому результату. Посмотрим что покажет ArcFace.

In [50]:
EMBEDDING_DIM = 512
NUM_CLASSES = 1000

class EmbeddingModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=1000):
        super().__init__()
        backbone = models.resnet50(pretrained=True)
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.embedding_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.4),
            nn.Linear(1024, embedding_dim)
        )

    def forward(self, x, return_embeddings=False):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        embeddings = self.embedding_head(x)

        if return_embeddings:
            return embeddings
        return embeddings

model_arcface = EmbeddingModel(EMBEDDING_DIM, NUM_CLASSES).to(DEVICE)
arcface_layer = ArcFaceLayer(EMBEDDING_DIM, NUM_CLASSES).to(DEVICE)

optimizer_arc = torch.optim.AdamW([
    {'params': model_arcface.parameters(), 'lr': 1e-3},
    {'params': arcface_layer.parameters(), 'lr': 5e-3}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_arc, max_lr=5e-3, epochs=15, steps_per_epoch=len(train_loader),
    pct_start=0.3, anneal_strategy='cos'
)

def train_epoch_arcface(model, arcface_layer, loader, optimizer):
    model.train()
    arcface_layer.train()
    total_loss = correct = total = 0
    pbar = tqdm(loader, desc="ArcFace Train")

    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        # Получаем embeddings
        embeddings = model(images, return_embeddings=True)

        # ArcFace loss
        logits = arcface_layer(embeddings, labels)
        loss = F.cross_entropy(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        pred = logits.argmax(1)
        correct += (pred == labels).sum().item()
        total += len(labels)
        pbar.set_postfix({'loss': f"{loss.item():.3f}", 'acc': f"{correct/total:.1%}"})

    return total_loss/len(loader), correct/total

def validate_arcface(model, arcface_layer, loader):
    model.eval()
    arcface_layer.eval()
    correct = total = 0
    with torch.no_grad():
        pbar = tqdm(loader, desc="ArcFace Val")
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            embeddings = model(images, return_embeddings=True)
            logits = arcface_layer(embeddings, labels)
            pred = logits.argmax(1)
            correct += (pred == labels).sum().item()
            total += len(labels)
            pbar.set_postfix({'acc': f"{correct/total:.1%}"})
    return correct/total

best_arc_acc = 0
for epoch in range(15):
    train_loss, train_acc = train_epoch_arcface(model_arcface, arcface_layer, train_loader, optimizer_arc)
    val_acc = validate_arcface(model_arcface, arcface_layer, val_loader)

    scheduler.step()

    print(f"\nEpoch {epoch+1}/15")
    print(f"  ArcFace Train: {train_acc:.1%} | Val: {val_acc:.1%}")
    print(f"  LR: {scheduler.get_last_lr()[0]:.1e}")

    if val_acc > best_arc_acc:
        best_arc_acc = val_acc
        torch.save(model_arcface.state_dict(), '/content/best_arcface_new.pth')
        torch.save(arcface_layer.state_dict(), '/content/arcface_layer_new.pth')
        print(f" BEST: {best_arc_acc:.1%} ")

    if best_arc_acc > 0.75:
        break

print(f"\nARCFACE ИТОГ: {best_arc_acc:.1%}")
torch.save(model_arcface.state_dict(), '/content/final_arcface_new.pth')
torch.save(arcface_layer.state_dict(), '/content/final_arcface_layer_new.pth')


ArcFace Val: 100%|██████████| 47/47 [00:06<00:00,  6.99it/s, acc=0.0%]



Epoch 1/15
  ArcFace Train: 0.0% | Val: 0.0%
  LR: 2.0e-04


ArcFace Val: 100%|██████████| 47/47 [00:07<00:00,  6.09it/s, acc=0.0%]



Epoch 2/15
  ArcFace Train: 0.0% | Val: 0.0%
  LR: 2.0e-04


ArcFace Val: 100%|██████████| 47/47 [00:08<00:00,  5.73it/s, acc=0.0%]


...


Epoch 10/15
  ArcFace Train: 15.0% | Val: 8.6%
  LR: 2.0e-04
 BEST: 8.6% 


ArcFace Val: 100%|██████████| 47/47 [00:07<00:00,  6.30it/s, acc=11.9%]



Epoch 11/15
  ArcFace Train: 21.1% | Val: 11.9%
  LR: 2.0e-04
 BEST: 11.9% 


ArcFace Train:  14%|█▍        | 26/187 [00:07<00:45,  3.53it/s, loss=6.420, acc=26.1%]


KeyboardInterrupt: 

Здесь вообще все печально, сетую на такие, как и ранее причины, видмо обучится при таких данных модели не получается, ну или я упускаю что-то очень значимое в своем решении.

Ну писать код сравнения с такими результатами смысла нет, да я и не успеваю уже.

По итогу успел сделать и описать более менее лишь первое задание, тут сколько бы не бился и не делал запросы куда угодно результат получил слабый. Так что второе задание не знаю как будет учитываться, могу лишь понадеяться, что получу заветные 5 баллов для диплома (за второе задание хоть что-то за попытку), жаль, что не получилось по семейным обстоятельствам и моей лени потратить больше времени на этот проект.

Не взирая на мое нытье, могу сказать только спасибо за ваши курсы, а за расписанные задания для проекта, который можно в резюме кинуть - вообще респект. Честно не было желания подлизать за баллы, правда благодарен за вашу работу. Удачи вам!